In [67]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib


In [68]:
# Load model artifact and ML feature table
# Needed to load pickled FunctionTransformer from training

def to_dense_fn(x):
    return x.toarray() if hasattr(x, "toarray") else x

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "ml_df.csv").exists() and (ROOT.parent / "data" / "ml_df.csv").exists():
    ROOT = ROOT.parent

artifact_path = ROOT / "models" / "worldcup_model.joblib"
artifact = joblib.load(artifact_path)

model = artifact["model"]
feature_cols = artifact["feature_cols"]
class_order = artifact["class_order"]
score_models = artifact.get("score_models", {})

ml_df = pd.read_csv(ROOT / "data" / "ml_df.csv")
if "date" in ml_df.columns:
    ml_df["date"] = pd.to_datetime(ml_df["date"], errors="coerce")

# Ensure ranking_diff exists if required by the model
if "ranking_diff" in feature_cols and "ranking_diff" not in ml_df.columns:
    if "home_ranking_score" in ml_df.columns and "away_ranking_score" in ml_df.columns:
        ml_df["ranking_diff"] = ml_df["home_ranking_score"] - ml_df["away_ranking_score"]
    else:
        ml_df["ranking_diff"] = np.nan

ml_df.head()


,year,date,tournament_id,tournament_name,match_name,stage,home_team,away_team,home_team_code,away_team_code,...,win_rate_diff,goal_diff_diff,form_diff,goals_per_match_diff,conceded_per_match_diff,season_win_rate_diff,season_goal_diff_diff,elo_diff,result_target,ranking_diff
0,1930,1930-07-13,WC-1930,1930 FIFA World Cup,France v Mexico,Group stage,France,Mexico,FRA,MEX,...,NaN,-3.0,NaN,NaN,NaN,NaN,-3.0,0.0,HomeWin,NaN
1,1930,1930-07-13,WC-1930,1930 FIFA World Cup,United States v Belgium,Group stage,United States,Belgium,USA,BEL,...,NaN,-6.0,NaN,NaN,NaN,NaN,-6.0,0.0,HomeWin,NaN
2,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Romania v Peru,Group stage,Romania,Peru,ROU,PER,...,NaN,-5.0,NaN,NaN,NaN,NaN,-5.0,0.0,HomeWin,NaN
3,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Yugoslavia v Brazil,Group stage,Yugoslavia,Brazil,YUG,BRA,...,NaN,-3.0,NaN,NaN,NaN,NaN,-3.0,0.0,HomeWin,NaN
4,1930,1930-07-15,WC-1930,1930 FIFA World Cup,Argentina v France,Group stage,Argentina,France,ARG,FRA,...,NaN,-2.0,NaN,NaN,NaN,NaN,-2.0,-10.0,HomeWin,NaN


In [69]:
# Pick matches from 2022: 2 per stage from Group to Semis, and 1 from the Final
ml_df_2022 = ml_df[ml_df["year"] == 2022].reset_index(drop=True)

stage_series = ml_df_2022["stage"].astype(str).str.lower()

# Helper to select N earliest matches for a stage pattern
def pick_stage(pattern, n):
    return ml_df_2022[stage_series.str.contains(pattern)].sort_values("date").head(n)

# Group stage (2)
group_matches = pick_stage("group", 2)

# Round of 16 (2) if available, otherwise Quarter-finals (2)
if (stage_series.str.contains("round of 16")).any():
    r16_matches = pick_stage("round of 16", 2)
else:
    r16_matches = pick_stage("quarter", 2)

# Quarter-finals (2) if present
quarter_matches = pick_stage("quarter", 2) if (stage_series.str.contains("quarter")).any() else pd.DataFrame([])

# Semi-finals (2) if present
semi_matches = pick_stage("semi", 2) if (stage_series.str.contains("semi")).any() else pd.DataFrame([])

# Final (1)
final_mask = stage_series.eq("final")
if final_mask.any():
    final_match = ml_df_2022[final_mask].sort_values("date").head(1)
else:
    final_match = ml_df_2022[stage_series.str.contains("final") & ~stage_series.str.contains("semi")].sort_values("date").head(1)

# Combine and keep unique matches
parts = [group_matches, r16_matches, quarter_matches, semi_matches, final_match]
test_rows = pd.concat([p for p in parts if len(p) > 0]).drop_duplicates().reset_index(drop=True)

test_rows[["date", "stage", "home_team", "away_team", "result_target"]]


,date,stage,home_team,away_team,result_target
0,2022-11-20,Group stage,Qatar,Ecuador,AwayWin
1,2022-11-21,Group stage,England,Iran,HomeWin
2,2022-12-03,round of 16,Argentina,Australia,HomeWin
3,2022-12-03,round of 16,Netherlands,United States,HomeWin
4,2022-12-09,quarter-finals,Croatia,Brazil,Draw
5,2022-12-09,quarter-finals,Netherlands,Argentina,Draw
6,2022-12-13,semi-finals,Argentina,Croatia,HomeWin
7,2022-12-14,semi-finals,France,Morocco,HomeWin
8,2022-12-18,final,Argentina,France,Draw


In [70]:
# Monte Carlo simulation for the three selected matches
results = []

score_home_model = score_models.get("home_goals")
score_away_model = score_models.get("away_goals")


def round_goal(x: float) -> int:
    if x is None or np.isnan(x):
        return 0
    return max(0, int(np.floor(x + 0.5)))


def mc_simulate_outcomes(proba, class_order, n=10000, seed=42):
    rng = np.random.default_rng(seed)
    sims = rng.choice(class_order, size=n, p=proba)
    counts = pd.Series(sims).value_counts().reindex(class_order, fill_value=0)
    return (counts / n).to_dict()


for i, row in test_rows.iterrows():
    X = pd.DataFrame([row]).reindex(columns=feature_cols)
    proba = model.predict_proba(X)[0]
    pred = class_order[int(np.argmax(proba))]
    actual = row["result_target"]

    mc_probs = mc_simulate_outcomes(proba, class_order, n=20000, seed=42 + i)
    mc_pred = max(mc_probs, key=mc_probs.get)

    pred_home_goals = None
    pred_away_goals = None
    if score_home_model is not None and score_away_model is not None:
        pred_home_goals = round_goal(float(score_home_model.predict(X)[0]))
        pred_away_goals = round_goal(float(score_away_model.predict(X)[0]))

    results.append({
        "date": row.get("date"),
        "stage": row.get("stage"),
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "actual": actual,
        "predicted": pred,
        "mc_predicted": mc_pred,
        "mc_prob_actual": mc_probs.get(actual, 0.0),
        "correct": pred == actual,
        "actual_home_goals": row.get("home_goals"),
        "actual_away_goals": row.get("away_goals"),
        "pred_home_goals": pred_home_goals,
        "pred_away_goals": pred_away_goals,
        "p_homewin": proba[class_order.index("HomeWin")],
        "p_draw": proba[class_order.index("Draw")],
        "p_awaywin": proba[class_order.index("AwayWin")],
        "mc_homewin": mc_probs.get("HomeWin", 0.0),
        "mc_draw": mc_probs.get("Draw", 0.0),
        "mc_awaywin": mc_probs.get("AwayWin", 0.0),
    })

results_df = pd.DataFrame(results)
results_df


,date,stage,home_team,away_team,actual,predicted,mc_predicted,mc_prob_actual,correct,actual_home_goals,actual_away_goals,pred_home_goals,pred_away_goals,p_homewin,p_draw,p_awaywin,mc_homewin,mc_draw,mc_awaywin
0,2022-11-20,Group stage,Qatar,Ecuador,AwayWin,HomeWin,HomeWin,0.00030,False,0.0,2.0,0,2,0.999192,0.000521,0.000287,0.99905,0.00065,0.00030
1,2022-11-21,Group stage,England,Iran,HomeWin,AwayWin,AwayWin,0.00030,False,6.0,2.0,5,2,0.000183,0.000483,0.999335,0.00030,0.00035,0.99935
2,2022-12-03,round of 16,Argentina,Australia,HomeWin,AwayWin,AwayWin,0.00075,False,2.0,1.0,1,1,0.000717,0.000724,0.998559,0.00075,0.00105,0.99820
3,2022-12-03,round of 16,Netherlands,United States,HomeWin,AwayWin,AwayWin,0.00190,False,3.0,1.0,2,1,0.001602,0.001063,0.997335,0.00190,0.00110,0.99700
4,2022-12-09,quarter-finals,Croatia,Brazil,Draw,HomeWin,HomeWin,0.01350,False,1.0,1.0,1,2,0.982954,0.014987,0.002059,0.98460,0.01350,0.00190
5,2022-12-09,quarter-finals,Netherlands,Argentina,Draw,Draw,Draw,0.81940,True,2.0,2.0,1,0,0.039916,0.816296,0.143788,0.03795,0.81940,0.14265
6,2022-12-13,semi-finals,Argentina,Croatia,HomeWin,AwayWin,AwayWin,0.01730,False,3.0,0.0,1,1,0.019079,0.229989,0.750933,0.01730,0.22915,0.75355
7,2022-12-14,semi-finals,France,Morocco,HomeWin,AwayWin,AwayWin,0.00045,False,2.0,0.0,2,1,0.000337,0.000666,0.998996,0.00045,0.00070,0.99885
8,2022-12-18,final,Argentina,France,Draw,Draw,Draw,0.82095,True,3.0,3.0,1,2,0.070446,0.820430,0.109125,0.07215,0.82095,0.10690


In [71]:
# Summarize Monte Carlo performance on the three selected matches
summary = pd.DataFrame({
    "total_tests": [len(results_df)],
    "correct_predictions": [results_df["correct"].sum()],
    "accuracy": [results_df["correct"].mean()],
    "avg_mc_prob_actual": [results_df["mc_prob_actual"].mean()],
})

summary


,total_tests,correct_predictions,accuracy,avg_mc_prob_actual
0,9,2,0.222222,0.186094


In [72]:
# Full 2022 evaluation (all matches)
ml_df_2022 = ml_df[ml_df["year"] == 2022].reset_index(drop=True)

X_2022 = ml_df_2022.reindex(columns=feature_cols)
y_2022 = ml_df_2022["result_target"]

proba_2022 = model.predict_proba(X_2022)
pred_2022 = [class_order[int(i)] for i in np.argmax(proba_2022, axis=1)]

full_results = pd.DataFrame({
    "actual": y_2022,
    "predicted": pred_2022,
})
full_results["correct"] = full_results["actual"] == full_results["predicted"]

full_accuracy = full_results["correct"].mean()
full_accuracy

0.234375

In [73]:
# Baseline: always predict HomeWin, then compare
baseline_pred = ["HomeWin"] * len(y_2022)
baseline_correct = (y_2022.values == np.array(baseline_pred))
baseline_accuracy = baseline_correct.mean()

comparison = pd.DataFrame({
    "model": ["trained_model", "baseline_homewin"],
    "accuracy": [full_accuracy, baseline_accuracy],
    "total_matches": [len(y_2022), len(y_2022)],
    "correct_predictions": [full_results["correct"].sum(), baseline_correct.sum()],
})

comparison

,model,accuracy,total_matches,correct_predictions
0,trained_model,0.234375,64,15
1,baseline_homewin,0.453125,64,29
